# Linear Regression Experiments


## Setup


In [1]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not find repo root (pyproject.toml). Open this notebook from the repo.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
REPO_ROOT


WindowsPath('C:/Users/baben_bakg1j1/HSE/annual_project/stocks-advisor')

In [2]:
import json
import tempfile
from copy import deepcopy
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from jupyter_utils import setup_jupyter_notebook
from stocks_dl.constants import TARGET_COLUMN
from stocks_dl.data.pipeline import load_features_multi
from stocks_dl.training.dataset import split_train_test
from stocks_dl.training.train import calculate_metrics

import warnings
warnings.filterwarnings('ignore')


In [3]:
EXPERIMENT_NAME = 'linear_regression_checkpoint'
setup_jupyter_notebook(environment='prod', experiment=EXPERIMENT_NAME)

# Для локального запуска:
# setup_jupyter_notebook(environment='local', experiment='linear_regression_test')


2026/06/09 13:38:10 INFO mlflow.tracking.fluent: Experiment with name 'linear_regression_checkpoint' does not exist. Creating a new experiment.


Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: linear_regression_checkpoint
Database: localhost:15432/stocks_advisor_db


In [4]:
TICKERS = ['SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN']
SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
ENRICHMENTS_LIMIT = 1_000_000

RUNS_DIR = REPO_ROOT / 'stocks_dl_runs' / 'linear_regression'
RUNS_DIR.mkdir(parents=True, exist_ok=True)


## Data


In [5]:
features_by_ticker, enrichments_df = load_features_multi(
    TICKERS,
    enrichments_limit=ENRICHMENTS_LIMIT,
)

for ticker, df in features_by_ticker.items():
    print(ticker, df.shape, df['begin'].min(), df['begin'].max())


SBER (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
TCSG (2853, 89) 2022-07-04 10:00:00 2024-11-20 18:00:00
GAZP (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
LKOH (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
ROSN (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00


## Helpers


In [6]:
def build_lreg_configs(ticker: str) -> list[dict]:
    return [
        {
            'run_name': f'{ticker.lower()}_lreg_01_linear',
            'ticker': ticker,
            'variant': 'linear',
            'model_class': LinearRegression,
            'model_params': {},
        },
        {
            'run_name': f'{ticker.lower()}_lreg_02_ridge_alpha01',
            'ticker': ticker,
            'variant': 'ridge_alpha01',
            'model_class': Ridge,
            'model_params': {'alpha': 0.1, 'random_state': SEED},
        },
        {
            'run_name': f'{ticker.lower()}_lreg_03_ridge_alpha10',
            'ticker': ticker,
            'variant': 'ridge_alpha10',
            'model_class': Ridge,
            'model_params': {'alpha': 10.0, 'random_state': SEED},
        },
        {
            'run_name': f'{ticker.lower()}_lreg_04_lasso_alpha001',
            'ticker': ticker,
            'variant': 'lasso_alpha001',
            'model_class': Lasso,
            'model_params': {'alpha': 0.01, 'max_iter': 20000, 'random_state': SEED},
        },
        {
            'run_name': f'{ticker.lower()}_lreg_05_elasticnet_alpha001_l1_05',
            'ticker': ticker,
            'variant': 'elasticnet_alpha001_l1_05',
            'model_class': ElasticNet,
            'model_params': {'alpha': 0.01, 'l1_ratio': 0.5, 'max_iter': 20000, 'random_state': SEED},
        },
    ]


def make_xy(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    feature_cols = [c for c in train_df.columns if c not in ('begin', TARGET_COLUMN)]

    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_val = val_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

    y_train = train_df[TARGET_COLUMN]
    y_val = val_df[TARGET_COLUMN]
    y_test = test_df[TARGET_COLUMN]

    return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols


def build_pipeline(model_class, model_params: dict) -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model_class(**model_params)),
    ])


def predictions_frame(test_df: pd.DataFrame, y_true, y_pred) -> pd.DataFrame:
    out = pd.DataFrame({
        'begin': pd.to_datetime(test_df['begin']).reset_index(drop=True),
        TARGET_COLUMN: np.asarray(y_true),
        'predict': np.asarray(y_pred),
    })
    out['error'] = out['predict'] - out[TARGET_COLUMN]
    out['abs_error'] = out['error'].abs()
    out['direction_match'] = np.sign(out['predict']) == np.sign(out[TARGET_COLUMN])
    return out


In [7]:
def run_lreg_experiment(
    cfg: dict,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = make_xy(train_df, val_df, test_df)

    params = deepcopy(cfg)
    run_name = params.pop('run_name')
    ticker = params.pop('ticker')
    variant = params.pop('variant')
    model_class = params.pop('model_class')
    model_params = params.pop('model_params')

    pipeline = build_pipeline(model_class, model_params)

    with tempfile.TemporaryDirectory() as tmpdir, mlflow.start_run(run_name=run_name) as run:
        tmpdir = Path(tmpdir)

        mlflow.set_tags({
            'ticker': ticker,
            'stage': 'linear_regression_search',
            'model_family': 'LinearRegression',
            'target': TARGET_COLUMN,
            'variant': variant,
            'seed': str(SEED),
        })
        mlflow.log_param('variant', variant)
        mlflow.log_param('model_class', model_class.__name__)
        mlflow.log_param('feature_count', len(feature_cols))
        for key, value in model_params.items():
            mlflow.log_param(key, value)
        mlflow.log_dict(
            {
                'run_name': run_name,
                'ticker': ticker,
                'variant': variant,
                'model_class': model_class.__name__,
                'model_params': model_params,
            },
            'config.json',
        )

        pipeline.fit(X_train, y_train)

        train_pred = pipeline.predict(X_train)
        val_pred = pipeline.predict(X_val)
        test_pred = pipeline.predict(X_test)

        train_metrics = calculate_metrics(y_train.to_numpy(), train_pred)
        val_metrics = calculate_metrics(y_val.to_numpy(), val_pred)
        test_metrics = calculate_metrics(y_test.to_numpy(), test_pred)

        pred_df = predictions_frame(test_df, y_test.to_numpy(), test_pred)
        pred_df.to_csv(tmpdir / 'test_predictions.csv', index=False)

        model = pipeline.named_steps['model']
        if hasattr(model, 'coef_'):
            coef_df = pd.DataFrame({
                'feature': feature_cols,
                'coef': np.asarray(model.coef_).reshape(-1),
                'abs_coef': np.abs(np.asarray(model.coef_).reshape(-1)),
            }).sort_values('abs_coef', ascending=False)
            coef_df.to_csv(tmpdir / 'coefficients.csv', index=False)

        mlflow.log_artifacts(str(tmpdir))
        mlflow.sklearn.log_model(pipeline, 'model')

        summary = {
            'run_id': run.info.run_id,
            'run_name': run_name,
            'ticker': ticker,
            'variant': variant,
            'model_class': model_class.__name__,
            'alpha': model_params.get('alpha', None),
            'l1_ratio': model_params.get('l1_ratio', None),
            'feature_count': len(feature_cols),
            'train_mae': float(train_metrics['mae']),
            'train_rmse': float(train_metrics['rmse']),
            'train_r2': float(train_metrics['r2']),
            'train_direction_accuracy': float(train_metrics['direction_accuracy']),
            'val_mae': float(val_metrics['mae']),
            'val_rmse': float(val_metrics['rmse']),
            'val_r2': float(val_metrics['r2']),
            'val_direction_accuracy': float(val_metrics['direction_accuracy']),
            'test_mae': float(test_metrics['mae']),
            'test_rmse': float(test_metrics['rmse']),
            'test_r2': float(test_metrics['r2']),
            'test_direction_accuracy': float(test_metrics['direction_accuracy']),
            'config_json': json.dumps(
                {
                    'run_name': run_name,
                    'ticker': ticker,
                    'variant': variant,
                    'model_class': model_class.__name__,
                    'model_params': model_params,
                },
                ensure_ascii=False,
            ),
        }
        mlflow.log_metrics({k: v for k, v in summary.items() if isinstance(v, (int, float, np.floating))})

    print(f'[{run_name}] val_dir_acc={summary["val_direction_accuracy"]:.4f} test_dir_acc={summary["test_direction_accuracy"]:.4f}')
    return summary


## Experiments


In [8]:
mlflow.set_experiment(EXPERIMENT_NAME)

all_rows = []
ticker_summaries = {}

for ticker in TICKERS:
    print(f'\n=== {ticker} ===')
    features_df = features_by_ticker[ticker].copy()
    train_df, val_df, test_df = split_train_test(
        features_df,
        target_column=TARGET_COLUMN,
        test_size=TEST_SIZE,
        val_size=VAL_SIZE,
    )

    rows = []
    for cfg in build_lreg_configs(ticker):
        rows.append(run_lreg_experiment(cfg, train_df, val_df, test_df))

    summary_df = (
        pd.DataFrame(rows)
        .sort_values(['val_direction_accuracy', 'val_r2', 'val_rmse'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    ticker_summaries[ticker] = summary_df
    all_rows.extend(rows)

    out_path = RUNS_DIR / f'{ticker.lower()}_lreg_summary.csv'
    summary_df.to_csv(out_path, index=False)
    display(summary_df)



=== SBER ===


2026/06/09 13:41:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:41:26 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:41:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 13:41:27 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:41:27 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements vi

🏃 View run sber_lreg_01_linear at: http://localhost:5050/#/experiments/6/runs/f9c95859e497405e840f32151e44df7c
🧪 View experiment at: http://localhost:5050/#/experiments/6
[sber_lreg_01_linear] val_dir_acc=0.5679 test_dir_acc=0.5286


2026/06/09 13:41:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:41:30 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:41:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 13:41:30 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:41:30 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements vi

🏃 View run sber_lreg_02_ridge_alpha01 at: http://localhost:5050/#/experiments/6/runs/0bc2016b66ed42a582db1418a8b2e792
🧪 View experiment at: http://localhost:5050/#/experiments/6
[sber_lreg_02_ridge_alpha01] val_dir_acc=0.5929 test_dir_acc=0.5381


...

2026/06/09 13:42:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:42:28 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:42:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 13:42:29 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:42:29 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements vi

🏃 View run rosn_lreg_04_lasso_alpha001 at: http://localhost:5050/#/experiments/6/runs/bd7eb395f5c84bc8a81fc8db6c059ef6
🧪 View experiment at: http://localhost:5050/#/experiments/6
[rosn_lreg_04_lasso_alpha001] val_dir_acc=0.4988 test_dir_acc=0.3743


2026/06/09 13:42:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:42:31 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:42:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 13:42:32 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:42:32 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements vi

🏃 View run rosn_lreg_05_elasticnet_alpha001_l1_05 at: http://localhost:5050/#/experiments/6/runs/65ca2f3d6dc64844a1d9c44271121bc0
🧪 View experiment at: http://localhost:5050/#/experiments/6
[rosn_lreg_05_elasticnet_alpha001_l1_05] val_dir_acc=0.5060 test_dir_acc=0.3695


,run_id,run_name,ticker,variant,model_class,alpha,l1_ratio,feature_count,train_mae,train_rmse,...,train_direction_accuracy,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy,config_json
0,0516995519f4462a83574db963fcba55,rosn_lreg_02_ridge_alpha01,ROSN,ridge_alpha01,Ridge,0.10,NaN,87,2.640123,3.683084,...,0.676295,4.080514,5.355143,-0.426159,0.533333,4.616204,5.986279,-1.200959,0.403810,"{""run_name"": ""rosn_lreg_02_ridge_alpha01"", ""ti..."
1,65ca2f3d6dc64844a1d9c44271121bc0,rosn_lreg_05_elasticnet_alpha001_l1_05,ROSN,elasticnet_alpha001_l1_05,ElasticNet,0.01,0.5,87,2.702382,3.871384,...,0.667957,3.863816,4.935110,-0.211210,0.505952,4.295953,5.505801,-0.861825,0.369524,"{""run_name"": ""rosn_lreg_05_elasticnet_alpha001..."
2,e3a68600060647d1946f4b7ad93b8c86,rosn_lreg_03_ridge_alpha10,ROSN,ridge_alpha10,Ridge,10.00,NaN,87,2.691571,3.818763,...,0.669744,3.914139,5.012386,-0.249438,0.503571,4.374929,5.621581,-0.940953,0.390476,"{""run_name"": ""rosn_lreg_03_ridge_alpha10"", ""ti..."
3,bd7eb395f5c84bc8a81fc8db6c059ef6,rosn_lreg_04_lasso_alpha001,ROSN,lasso_alpha001,Lasso,0.01,NaN,87,2.694613,3.853476,...,0.666468,3.872821,4.962566,-0.224724,0.498810,4.290789,5.491290,-0.852025,0.374286,"{""run_name"": ""rosn_lreg_04_lasso_alpha001"", ""t..."
4,f57cf631f06f4ec2ad5b8a096284ea01,rosn_lreg_01_linear,ROSN,linear,LinearRegression,NaN,NaN,87,2.626617,3.654033,...,0.672424,4.357759,5.738852,-0.637856,0.496429,4.855921,6.350757,-1.477131,0.391429,"{""run_name"": ""rosn_lreg_01_linear"", ""ticker"": ..."


## Best models


In [9]:
lreg_runs_summary_df = (
    pd.DataFrame(all_rows)
    .sort_values(['ticker', 'val_direction_accuracy', 'val_r2', 'val_rmse'], ascending=[True, False, False, True])
    .reset_index(drop=True)
)
lreg_runs_summary_df.to_csv(RUNS_DIR / 'lreg_runs_summary.csv', index=False)
lreg_runs_summary_df


,run_id,run_name,ticker,variant,model_class,alpha,l1_ratio,feature_count,train_mae,train_rmse,...,train_direction_accuracy,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy,config_json
0,37da1dea1a834cc3b35e8145989da561,gazp_lreg_01_linear,GAZP,linear,LinearRegression,NaN,NaN,87,3.317097,4.546956,...,0.606611,4.305403,5.475854,0.170740,0.657143,3.738214,4.437459,-1.257417,0.511429,"{""run_name"": ""gazp_lreg_01_linear"", ""ticker"": ..."
1,5923bc2d362e48838d52e4487ab424a7,gazp_lreg_02_ridge_alpha01,GAZP,ridge_alpha01,Ridge,0.10,NaN,87,3.309402,4.573961,...,0.609589,4.213228,5.361721,0.204948,0.635714,3.533303,4.240673,-1.061638,0.527619,"{""run_name"": ""gazp_lreg_02_ridge_alpha01"", ""ti..."
2,fc1b0cbd9c9c4f939ab93f5eed038596,gazp_lreg_04_lasso_alpha001,GAZP,lasso_alpha001,Lasso,0.01,NaN,87,3.403418,4.883394,...,0.579512,4.385755,5.457191,0.176383,0.590476,2.709565,3.416868,-0.338442,0.589524,"{""run_name"": ""gazp_lreg_04_lasso_alpha001"", ""t..."
3,bc5788ddc0974e4e9b06b114c316ef5a,gazp_lreg_03_ridge_alpha10,GAZP,ridge_alpha10,Ridge,10.00,NaN,87,3.427536,4.947491,...,0.586957,4.435154,5.513215,0.159386,0.578571,2.491797,3.222763,-0.190693,0.640000,"{""run_name"": ""gazp_lreg_03_ridge_alpha10"", ""ti..."
4,cf982c1451a0474cb668cf48fa9ff2c6,gazp_lreg_05_elasticnet_alpha001_l1_05,GAZP,elasticnet_alpha001_l1_05,ElasticNet,0.01,0.5,87,3.461343,5.061820,...,0.579214,4.526510,5.613877,0.128409,0.565476,2.372893,3.103900,-0.104481,0.642857,"{""run_name"": ""gazp_lreg_05_elasticnet_alpha001..."
5,8d21ce0e5cf44d2c8d77b93a996b91a0,lkoh_lreg_01_linear,LKOH,linear,LinearRegression,NaN,NaN,87,2.561993,3.394314,...,0.647707,2.762638,3.375324,0.103254,0.657143,3.435967,4.468554,-0.279239,0.460952,"{""run_name"": ""lkoh_lreg_01_linear"", ""ticker"": ..."
6,12dac1ca38444838b2aa18bcb9d6eb2c,lkoh_lreg_02_ridge_alpha01,LKOH,ridge_alpha01,Ridge,0.10,NaN,87,2.581674,3.430680,...,0.635795,2.771209,3.366191,0.108100,0.645238,3.462830,4.483669,-0.287908,0.422857,"{""run_name"": ""lkoh_lreg_02_ridge_alpha01"", ""ti..."
7,7d877be2c866419e8c859c0dc71b493b,lkoh_lreg_04_lasso_alpha001,LKOH,lasso_alpha001,Lasso,0.01,NaN,87,2.609933,3.507248,...,0.620905,2.749213,3.382380,0.099501,0.613095,3.288844,4.275043,-0.170843,0.396190,"{""run_name"": ""lkoh_lreg_04_lasso_alpha001"", ""t..."
8,690159a3cc6b44d0a66bc33426ac0ab0,lkoh_lreg_05_elasticnet_alpha001_l1_05,LKOH,elasticnet_alpha001_l1_05,ElasticNet,0.01,0.5,87,2.614760,3.510919,...,0.619416,2.749351,3.382617,0.099375,0.608333,3.281868,4.271643,-0.168982,0.393333,"{""run_name"": ""lkoh_lreg_05_elasticnet_alpha001..."
9,d1e24c382a494cbcb20119d8fcd22df0,lkoh_lreg_03_ridge_alpha10,LKOH,ridge_alpha10,Ridge,10.00,NaN,87,2.604128,3.491200,...,0.629839,2.752461,3.360617,0.111052,0.607143,3.330967,4.328179,-0.200130,0.400000,"{""run_name"": ""lkoh_lreg_03_ridge_alpha10"", ""ti..."


In [10]:
best_models_rows = []

for ticker, summary_df in ticker_summaries.items():
    best_row = summary_df.sort_values(
        ['val_direction_accuracy', 'val_r2', 'val_rmse'],
        ascending=[False, False, True],
    ).iloc[0]
    best_models_rows.append({
        'ticker': ticker,
        'run_name': best_row['run_name'],
        'run_id': best_row['run_id'],
        'variant': best_row['variant'],
        'model_class': best_row['model_class'],
        'alpha': best_row['alpha'],
        'l1_ratio': best_row['l1_ratio'],
        'feature_count': best_row['feature_count'],
        'val_mae': best_row['val_mae'],
        'val_rmse': best_row['val_rmse'],
        'val_r2': best_row['val_r2'],
        'val_direction_accuracy': best_row['val_direction_accuracy'],
        'test_mae': best_row['test_mae'],
        'test_rmse': best_row['test_rmse'],
        'test_r2': best_row['test_r2'],
        'test_direction_accuracy': best_row['test_direction_accuracy'],
    })

best_models_summary_df = pd.DataFrame(best_models_rows).sort_values('ticker').reset_index(drop=True)
best_models_summary_df.to_csv(RUNS_DIR / 'lreg_best_models_summary.csv', index=False)
best_models_summary_df


,ticker,run_name,run_id,variant,model_class,alpha,l1_ratio,feature_count,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy
0,GAZP,gazp_lreg_01_linear,37da1dea1a834cc3b35e8145989da561,linear,LinearRegression,NaN,NaN,87,4.305403,5.475854,0.170740,0.657143,3.738214,4.437459,-1.257417,0.511429
1,LKOH,lkoh_lreg_01_linear,8d21ce0e5cf44d2c8d77b93a996b91a0,linear,LinearRegression,NaN,NaN,87,2.762638,3.375324,0.103254,0.657143,3.435967,4.468554,-0.279239,0.460952
2,ROSN,rosn_lreg_02_ridge_alpha01,0516995519f4462a83574db963fcba55,ridge_alpha01,Ridge,0.10,NaN,87,4.080514,5.355143,-0.426159,0.533333,4.616204,5.986279,-1.200959,0.403810
3,SBER,sber_lreg_02_ridge_alpha01,0bc2016b66ed42a582db1418a8b2e792,ridge_alpha01,Ridge,0.10,NaN,87,2.526999,3.176813,-0.296696,0.592857,1.554124,2.130255,-0.304183,0.538095
4,TCSG,tcsg_lreg_04_lasso_alpha001,2692724ab0fd478ca7c4faf8d6e312cf,lasso_alpha001,Lasso,0.01,NaN,87,2.815133,3.711356,-0.583629,0.514223,5.327448,6.471031,-0.790416,0.404553
